# SkyOps Week 3 — Databricks Data Exploration Notebook

**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Week:** 3 — Data Exploration and Databricks Foundation  
**Databricks Volume:** `/Volumes/p07-skyops/default/skyops`

This notebook is the project-specific Week-3 conversion of the provided Week-3B learning material.

## Week-3 mission

Understand the four controlled batch source files before building the production Bronze/Silver/Gold pipeline:

- `flights.csv` — central flight-occurrence source
- `airports.csv` — airport reference data
- `carriers.csv` — carrier reference data
- `routes.csv` — directed origin-destination reference data

The notebook checks source availability, schemas, grain, keys, counts, distributions, ranges, missing values, suspicious records, relationships, one business question, and one Bronze demonstration table plus one lineage demonstration view.

## Week-3 boundary

This notebook **does not** build the full Bronze layer, Silver layer, Gold metrics, or streaming pipeline. Streaming is simulated and belongs to Week 10 according to the project data pack.

## Authoritative project inputs

This notebook was prepared from the supplied SkyOps project pack and the supplied Week-3B conversion material.

The project pack states that the batch files are first used in Week 2 for understanding and Week 3 for Databricks exploration. It also defines the four batch source files and their documented grains/keys.

**Important:** Do not replace the controlled project files with invented sample data or unrelated airline datasets.

## 1. Databricks language choice

Use **Spark SQL as the primary exploration language**. PySpark is kept for file loading, DataFrame creation, display, schema inspection, and temporary-view creation.

In [ ]:
from pyspark.sql import functions as F

volume_path = "/Volumes/p07-skyops/default/skyops"

source_files = {
    "flights": f"{volume_path}/flights.csv",
    "airports": f"{volume_path}/airports.csv",
    "carriers": f"{volume_path}/carriers.csv",
    "routes": f"{volume_path}/routes.csv",
}

print("Configured Week-3 source files:")
for name, path in source_files.items():
    print(f"- {name}: {path}")

## 2. Check the uploaded files

Before reading data, confirm that the four required files are present in the project Volume.

In [ ]:
%fs ls /Volumes/p07-skyops/default/skyops

### Expected Week-3 batch files

```text
flights.csv
airports.csv
carriers.csv
routes.csv
```

If any required file is missing, stop and upload the correct project file before continuing.

## 3. Read the four source files

Create one Spark DataFrame for each real project source. Schema inference is used only for Week-3 exploration; production schema enforcement belongs to later pipeline work.

In [ ]:
flights_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_files["flights"])
)

airports_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_files["airports"])
)

carriers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_files["carriers"])
)

routes_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_files["routes"])
)

print("DataFrames created:")
print("flights_df, airports_df, carriers_df, routes_df")

## 4. Inspect the flights DataFrame

In [ ]:
print("Schema: flights_df")
flights_df.printSchema()

print("Sample rows: flights_df")
display(flights_df)

In [ ]:
flights_df.createOrReplaceTempView("flights")
print("Temporary SQL view created: flights")

## 5. Inspect the airports DataFrame

In [ ]:
print("Schema: airports_df")
airports_df.printSchema()

print("Sample rows: airports_df")
display(airports_df)

In [ ]:
airports_df.createOrReplaceTempView("airports")
print("Temporary SQL view created: airports")

## 6. Inspect the carriers DataFrame

In [ ]:
print("Schema: carriers_df")
carriers_df.printSchema()

print("Sample rows: carriers_df")
display(carriers_df)

In [ ]:
carriers_df.createOrReplaceTempView("carriers")
print("Temporary SQL view created: carriers")

## 7. Inspect the routes DataFrame

In [ ]:
print("Schema: routes_df")
routes_df.printSchema()

print("Sample rows: routes_df")
display(routes_df)

In [ ]:
routes_df.createOrReplaceTempView("routes")
print("Temporary SQL view created: routes")

## 8. Understand source grain and keys

The supplied project data dictionary/source manifest defines these working grains:

| Source | Grain | Business/primary key |
|---|---|---|
| `flights.csv` | one scheduled flight occurrence / physical source record | `source_record_key` |
| `airports.csv` | one approved airport code | `airport_code` |
| `carriers.csv` | one reporting carrier code | `carrier_code` |
| `routes.csv` | one directed origin-destination pair | `route_id` |

`flights` is the central fact-like source. Airports, carriers and routes provide reference context.

## 9. Physical row counts versus distinct keys

This check establishes whether the documented key is unique in each source.

In [ ]:
%sql
SELECT
  'flights' AS source,
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT source_record_key) AS distinct_keys,
  COUNT(*) - COUNT(DISTINCT source_record_key) AS duplicate_key_rows
FROM flights

UNION ALL

SELECT
  'airports',
  COUNT(*),
  COUNT(DISTINCT airport_code),
  COUNT(*) - COUNT(DISTINCT airport_code)
FROM airports

UNION ALL

SELECT
  'carriers',
  COUNT(*),
  COUNT(DISTINCT carrier_code),
  COUNT(*) - COUNT(DISTINCT carrier_code)
FROM carriers

UNION ALL

SELECT
  'routes',
  COUNT(*),
  COUNT(DISTINCT route_id),
  COUNT(*) - COUNT(DISTINCT route_id)
FROM routes
ORDER BY source;

### Interpretation

A zero duplicate-key count supports the documented key uniqueness. The actual values must be taken from the executed result rather than written into the notebook in advance.

In [ ]:
%sql
SELECT
  source_record_key,
  COUNT(*) AS occurrences
FROM flights
GROUP BY source_record_key
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

## 10. Inspect category distributions

Category distributions help us understand how the source is populated before downstream modeling.

In [ ]:
%sql
SELECT
  reporting_carrier,
  COUNT(*) AS flight_rows
FROM flights
GROUP BY reporting_carrier
ORDER BY flight_rows DESC;

In [ ]:
%sql
SELECT
  cancelled_flag,
  COUNT(*) AS flight_rows
FROM flights
GROUP BY cancelled_flag
ORDER BY cancelled_flag;

In [ ]:
%sql
SELECT
  diverted_flag,
  COUNT(*) AS flight_rows
FROM flights
GROUP BY diverted_flag
ORDER BY diverted_flag;

## 11. Inspect date and numeric ranges

In [ ]:
%sql
SELECT
  MIN(flight_date) AS minimum_flight_date,
  MAX(flight_date) AS maximum_flight_date,
  COUNT(DISTINCT flight_date) AS distinct_flight_dates
FROM flights;

In [ ]:
%sql
SELECT
  MIN(distance_miles) AS minimum_distance_miles,
  MAX(distance_miles) AS maximum_distance_miles,
  AVG(distance_miles) AS average_distance_miles,
  MIN(departure_delay_minutes) AS minimum_departure_delay_minutes,
  MAX(departure_delay_minutes) AS maximum_departure_delay_minutes,
  AVG(departure_delay_minutes) AS average_departure_delay_minutes,
  MIN(arrival_delay_minutes) AS minimum_arrival_delay_minutes,
  MAX(arrival_delay_minutes) AS maximum_arrival_delay_minutes,
  AVG(arrival_delay_minutes) AS average_arrival_delay_minutes
FROM flights;

## 12. Inspect missing values

Nulls are not automatically errors. In this project, some operational fields are legitimately unavailable for cancelled or otherwise incomplete flight records. The purpose here is to measure the pattern before later quality handling.

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN source_record_key IS NULL THEN 1 ELSE 0 END) AS null_source_record_key,
  SUM(CASE WHEN flight_date IS NULL THEN 1 ELSE 0 END) AS null_flight_date,
  SUM(CASE WHEN reporting_carrier IS NULL THEN 1 ELSE 0 END) AS null_reporting_carrier,
  SUM(CASE WHEN flight_number IS NULL THEN 1 ELSE 0 END) AS null_flight_number,
  SUM(CASE WHEN tail_number IS NULL THEN 1 ELSE 0 END) AS null_tail_number,
  SUM(CASE WHEN origin_airport_code IS NULL THEN 1 ELSE 0 END) AS null_origin_airport_code,
  SUM(CASE WHEN destination_airport_code IS NULL THEN 1 ELSE 0 END) AS null_destination_airport_code,
  SUM(CASE WHEN actual_departure_hhmm IS NULL THEN 1 ELSE 0 END) AS null_actual_departure_hhmm,
  SUM(CASE WHEN actual_arrival_hhmm IS NULL THEN 1 ELSE 0 END) AS null_actual_arrival_hhmm,
  SUM(CASE WHEN departure_delay_minutes IS NULL THEN 1 ELSE 0 END) AS null_departure_delay_minutes,
  SUM(CASE WHEN arrival_delay_minutes IS NULL THEN 1 ELSE 0 END) AS null_arrival_delay_minutes,
  SUM(CASE WHEN cancellation_code IS NULL THEN 1 ELSE 0 END) AS null_cancellation_code
FROM flights;

### Reference-file null checks

In [ ]:
%sql
SELECT
  'airports' AS source,
  SUM(CASE WHEN airport_code IS NULL THEN 1 ELSE 0 END) AS null_key,
  SUM(CASE WHEN airport_name IS NULL THEN 1 ELSE 0 END) AS null_name
FROM airports

UNION ALL

SELECT
  'carriers',
  SUM(CASE WHEN carrier_code IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN carrier_name IS NULL THEN 1 ELSE 0 END)
FROM carriers

UNION ALL

SELECT
  'routes',
  SUM(CASE WHEN route_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN route_label IS NULL THEN 1 ELSE 0 END)
FROM routes;

## 13. Look for suspicious operational values

Week 3 records observations; it does not clean or correct them.

In [ ]:
%sql
SELECT
  SUM(CASE WHEN departure_delay_minutes < 0 THEN 1 ELSE 0 END) AS negative_departure_delay_rows,
  SUM(CASE WHEN arrival_delay_minutes < 0 THEN 1 ELSE 0 END) AS negative_arrival_delay_rows,
  SUM(CASE WHEN cancelled_flag = 1 AND actual_departure_hhmm IS NOT NULL THEN 1 ELSE 0 END) AS cancelled_with_departure_time_rows,
  SUM(CASE WHEN cancelled_flag = 1 AND actual_arrival_hhmm IS NOT NULL THEN 1 ELSE 0 END) AS cancelled_with_arrival_time_rows,
  SUM(CASE WHEN diverted_flag = 1 AND cancelled_flag = 1 THEN 1 ELSE 0 END) AS diverted_and_cancelled_rows
FROM flights;

### Display examples of suspicious records

In [ ]:
%sql
SELECT
  source_record_key,
  flight_date,
  reporting_carrier,
  origin_airport_code,
  destination_airport_code,
  departure_delay_minutes,
  arrival_delay_minutes,
  cancelled_flag,
  diverted_flag,
  actual_departure_hhmm,
  actual_arrival_hhmm
FROM flights
WHERE arrival_delay_minutes < 0
   OR departure_delay_minutes < 0
   OR (cancelled_flag = 1 AND actual_departure_hhmm IS NOT NULL)
   OR (cancelled_flag = 1 AND actual_arrival_hhmm IS NOT NULL)
LIMIT 20;

## 🧠 Intern checkpoint 1

Choose one observed data concern and complete:

```text
Issue:
Evidence from the query:
Why it matters:
Should it be handled in exploration, Bronze, Silver/DQ, or later:
```

Do not silently fix the record in this notebook.

## 14. Validate flight-to-airport relationships

Every flight origin and destination should exist in the approved airport reference file.

In [ ]:
%sql
SELECT COUNT(*) AS invalid_origin_airport_references
FROM flights f
LEFT ANTI JOIN airports a
  ON f.origin_airport_code = a.airport_code;

In [ ]:
%sql
SELECT COUNT(*) AS invalid_destination_airport_references
FROM flights f
LEFT ANTI JOIN airports a
  ON f.destination_airport_code = a.airport_code;

## 15. Validate flight-to-carrier relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_carrier_references
FROM flights f
LEFT ANTI JOIN carriers c
  ON f.reporting_carrier = c.carrier_code;

## 16. Validate route reference data

First check that every route's origin and destination airports are valid. Then check whether flight origin-destination pairs exist in the controlled route reference.

In [ ]:
%sql
SELECT COUNT(*) AS invalid_route_origin_airport_references
FROM routes r
LEFT ANTI JOIN airports a
  ON r.origin_airport_code = a.airport_code;

In [ ]:
%sql
SELECT COUNT(*) AS invalid_route_destination_airport_references
FROM routes r
LEFT ANTI JOIN airports a
  ON r.destination_airport_code = a.airport_code;

In [ ]:
%sql
SELECT COUNT(*) AS flights_without_route_reference
FROM flights f
LEFT ANTI JOIN routes r
  ON f.origin_airport_code = r.origin_airport_code
 AND f.destination_airport_code = r.destination_airport_code;

### Why the route check is written this way

The route table is keyed by `route_id`, but flights do not carry `route_id`. Therefore the meaningful flight-to-route relationship for Week 3 is the directed pair `(origin_airport_code, destination_airport_code)`.

In [ ]:
%sql
SELECT
  f.origin_airport_code,
  f.destination_airport_code,
  COUNT(*) AS flight_rows
FROM flights f
LEFT ANTI JOIN routes r
  ON f.origin_airport_code = r.origin_airport_code
 AND f.destination_airport_code = r.destination_airport_code
GROUP BY f.origin_airport_code, f.destination_airport_code
ORDER BY flight_rows DESC
LIMIT 20;

## 17. Understand the effect of an inner join

An inner join keeps only records that have a matching reference. Compare the source flight count with the number of flights that match a carrier and both airports.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM flights) AS source_flight_rows,
  (
    SELECT COUNT(*)
    FROM flights f
    INNER JOIN carriers c
      ON f.reporting_carrier = c.carrier_code
    INNER JOIN airports ao
      ON f.origin_airport_code = ao.airport_code
    INNER JOIN airports ad
      ON f.destination_airport_code = ad.airport_code
  ) AS fully_matched_flight_rows;

## 18. Ask one business question

**Question:** Which reporting carriers have the highest average arrival delay in the controlled January 2018 source?

This is an exploratory result. It should not be treated as a trusted Gold KPI because Week 3 has not performed production quality transformations.

In [ ]:
%sql
SELECT
  reporting_carrier,
  COUNT(*) AS flight_rows,
  AVG(arrival_delay_minutes) AS average_arrival_delay_minutes
FROM flights
GROUP BY reporting_carrier
ORDER BY average_arrival_delay_minutes DESC;

### Intern checkpoint 2

After running the query, record:

```text
Highest average arrival-delay carrier:
Average arrival delay:
Number of flight rows:
One-line observation:
One limitation:
```

## 19. Create the Week-3 Bronze demonstration table

The Week-3 material requires a **single demonstration Bronze table**, not the complete production Bronze layer.

Use the central `flights` source because it is the main transaction/fact-like entity.

The demonstration table:

- preserves the source columns;
- adds `ingested_at`;
- records the source file path in `source_file`;
- does not deduplicate;
- does not clean;
- does not create Silver/Gold transformations.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.skyops_week03_bronze_demo_flights
USING DELTA
AS
SELECT
  *,
  current_timestamp() AS ingested_at,
  '/Volumes/p07-skyops/default/skyops/flights.csv' AS source_file
FROM flights;

## 20. Confirm and preview the Bronze demonstration table

In [ ]:
%sql
SHOW TABLES IN workspace.default LIKE 'skyops_week03_bronze_demo_flights';

In [ ]:
%sql
SELECT *
FROM workspace.default.skyops_week03_bronze_demo_flights
LIMIT 10;

## 21. Source-to-Bronze reconciliation

For this Week-3 demonstration, the Bronze table should contain the same number of rows as the source because no deduplication or filtering is performed.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM flights) AS source_rows,
  (SELECT COUNT(*) FROM workspace.default.skyops_week03_bronze_demo_flights) AS bronze_demo_rows,
  (SELECT COUNT(*) FROM flights)
    -
  (SELECT COUNT(*) FROM workspace.default.skyops_week03_bronze_demo_flights)
    AS row_count_difference;

### Interpretation

A `row_count_difference` of zero demonstrates that the Week-3 Bronze preview preserved the source row count.

In [ ]:
%sql
SELECT
  COUNT(*) AS bronze_rows,
  COUNT(DISTINCT source_record_key) AS distinct_source_keys,
  COUNT(*) - COUNT(DISTINCT source_record_key) AS duplicate_source_key_rows
FROM workspace.default.skyops_week03_bronze_demo_flights;

## 22. Inspect Delta table details

In [ ]:
%sql
DESCRIBE DETAIL workspace.default.skyops_week03_bronze_demo_flights;

## 23. Inspect Delta table history

In [ ]:
%sql
DESCRIBE HISTORY workspace.default.skyops_week03_bronze_demo_flights;

### Concepts

| Concept | What it helps answer |
|---|---|
| Schema | What columns and data types exist? |
| Relationship | How do business entities connect? |
| Delta detail | What is the physical/managed table metadata? |
| Delta history | What operations changed the Delta table? |
| Lineage | Which governed objects feed or use another object? |

## 24. Create one lineage demonstration view

The downstream view is intentionally simple so the lineage relationship is easy to explain.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.default.skyops_week03_lineage_demo_view
AS
SELECT
  source_record_key,
  reporting_carrier,
  origin_airport_code,
  destination_airport_code,
  flight_date,
  arrival_delay_minutes,
  cancelled_flag,
  diverted_flag,
  ingested_at
FROM workspace.default.skyops_week03_bronze_demo_flights;

In [ ]:
%sql
SELECT *
FROM workspace.default.skyops_week03_lineage_demo_view
LIMIT 20;

## 25. Understand the lineage flow

```text
Volume
  ↓
flights.csv
  ↓
flights DataFrame
  ↓
temporary SQL view: flights
  ↓
skyops_week03_bronze_demo_flights
  ↓
skyops_week03_lineage_demo_view
```

**Important distinction:**

- A **business relationship** connects entities such as flights → airports.
- **Delta history** records operations performed on a Delta table.
- **Data lineage** shows how governed data objects flow into downstream objects.

## 26. View lineage in Databricks Catalog Explorer

After the table and view are created:

1. Open **Catalog**.
2. Open `workspace`.
3. Open `default`.
4. Select `skyops_week03_lineage_demo_view`.
5. Open **Lineage**.
6. Open the lineage graph when available.
7. Confirm `skyops_week03_bronze_demo_flights` is upstream.
8. Capture the genuine lineage screenshot for the Week-3 evidence folder.

Do not create a screenshot before the objects actually exist.

## 27. Week-3 evidence checklist

Capture evidence after successful execution:

```text
screenshots/week03_01_source_files.png
screenshots/week03_02_dataframes.png
screenshots/week03_03_schemas.png
screenshots/week03_04_grain_counts_values.png
screenshots/week03_05_relationship_checks.png
screenshots/week03_06_business_question.png
screenshots/week03_07_bronze_demo.png
screenshots/week03_08_delta_history.png
screenshots/week03_09_lineage_graph.png
```

The screenshot contents must come from the actual Databricks run.

## 28. Week-3 boundary check

### Completed in this notebook

- real project files inspected;
- PySpark DataFrames created;
- DataFrames displayed;
- temporary SQL views created;
- schemas inspected;
- grain and keys documented;
- physical row counts and distinct-key counts checked;
- distributions and ranges explored;
- missing values measured;
- suspicious values inspected;
- airport, carrier and route relationships tested;
- one business question explored;
- exactly one Bronze demonstration table created;
- source-to-Bronze row count reconciled;
- Delta detail inspected;
- Delta history inspected;
- one downstream lineage demonstration view created.

### Deferred to later weeks

- full production Bronze layer;
- repeatable ingestion controls;
- full reconciliation framework;
- Silver transformations;
- data-quality quarantine/trusted Silver;
- Gold metrics and dashboards;
- streaming pipeline and Week-10 staged JSON drops.

## 29. AI transparency note

Complete this after verification:

```text
AI tool used:
Purpose:
Project files provided:
Week-3 material provided:
Source filenames manually verified:
Columns manually verified:
Grain manually verified:
Keys manually verified:
Relationships manually verified:
Notebook sections changed after testing:
Errors found during execution:
Corrections made:
Screenshots captured:
What I can explain without AI:
```

## 30. Final intern defence questions

Every intern should be able to answer:

1. What is the grain of each source file?
2. What is the business key of each source?
3. Why is `flights` the central source?
4. How do flights relate to airports and carriers?
5. How is the flight-to-route relationship established?
6. What is the difference between physical rows and distinct keys?
7. Which nulls are observations rather than automatic errors?
8. Why should Week 3 observe rather than clean the data?
9. What does the Bronze demonstration table prove?
10. Why is only one Bronze demonstration table created here?
11. What does `DESCRIBE DETAIL` show?
12. What does `DESCRIBE HISTORY` show?
13. What is data lineage?
14. How is lineage different from a business relationship?
15. What work is intentionally deferred to Week 4 and later?

# 🎉 Week-3 SkyOps notebook complete

```text
INSPECT
→ LOAD
→ VIEW
→ UNDERSTAND
→ CHECK
→ EXPLORE
→ DEMONSTRATE BRONZE
→ TRACE LINEAGE
→ VERIFY
→ EXPLAIN
```

**Stopping point:** one project Bronze demonstration table + one project lineage demonstration view.

The notebook becomes valid project evidence only after it is executed and verified in the assigned Databricks environment.